In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,BatchNormalization,Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model.

In [4]:
# this is the augmentation configuration we will use for training
batch_size=24
train_datagen = ImageDataGenerator(
        rescale=1./255,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True)

# this is the augmentation configuration we will use for testing:
# only rescaling
test_datagen = ImageDataGenerator(rescale=1./255)

# this is a generator that will read pictures found in
# subfolers of '/train', and indefinitely generate
# batches of augmented image data
train_generator = train_datagen.flow_from_directory(
        '../data/cats_and_dogs/train',  # this is the target directory
        target_size=(150, 150),  # all images will be resized to 150x150
        batch_size=batch_size,
        class_mode='binary')  # since we use binary_crossentropy loss, we need binary labels

# this is a similar generator, for validation data
validation_generator = test_datagen.flow_from_directory(
        '../data/cats_and_dogs/test',
        target_size=(150, 150),
        batch_size=batch_size,
        class_mode='binary')


Found 557 images belonging to 2 classes.
Found 140 images belonging to 2 classes.


In [8]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.regularizers import l2

In [9]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
# Freeze the convolutional layers to prevent updating their weights during training
for layer in base_model.layers:
    layer.trainable = False
# Create a new model on top of the pre-trained base model
model = Sequential()
model.add(base_model)
model.add(Flatten())
model.add(Dense(128,activation='relu',kernel_regularizer=l2(0.01)))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(64,activation='relu',kernel_regularizer=l2(0.01)))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(1,activation='sigmoid',kernel_regularizer=l2(0.01)))
model.summary()

     

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,772,481 (60.17 MB)

 Trainable params: 1,057,409 (4.03 MB)

 Non-trainable params: 14,715,072 (56.13 MB)

In [10]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [11]:
history = model.fit(train_generator,epochs=10,validation_data=validation_generator)

/home/the-ape/CurrentProjects/whatsapp-business/DSAI-course/DSAI_env/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 129s 5s/step - accuracy: 0.6928 - loss: 3.6903 - val_accuracy: 0.7214 - val_loss: 2.9381
Epoch 2/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 123s 5s/step - accuracy: 0.8276 - loss: 2.6952 - val_accuracy: 0.6357 - val_loss: 2.8304
Epoch 3/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 120s 5s/step - accuracy: 0.8829 - loss: 2.2222 - val_accuracy: 0.7357 - val_loss: 2.2885
Epoch 4/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 120s 5s/step - accuracy: 0.8560 - loss: 1.9550 - val_accuracy: 0.7214 - val_loss: 2.0739
Epoch 5/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 122s 5s/step - accuracy: 0.8873 - loss: 1.6492 - val_accuracy: 0.6786 - val_loss: 1.9786
Epoch 6/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 120s 5s/step - accuracy: 0.9184 - loss: 1.4125 - val_accuracy: 0.7214 - val_loss: 1.6912
Epoch 7/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 121s 5s/step - accuracy: 0.9094 - loss: 1.3468 - val_accuracy: 0.7429 - val_loss: 1.6502
Epoch 8/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 121s 5s/step - accuracy: 0.9305 - loss: 1.2010 - val_accuracy: 0.7500 - v

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'],color='red',label='train')
plt.plot(history.history['val_accuracy'],color='blue',label='validation')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'],color='red',label='train')
plt.plot(history.history['val_loss'],color='blue',label='validation')
plt.legend()
plt.show()